# Indicadores Econômicos do Brasil (2015–2024)
### Projeto G1 — Análise e Visualização de Dados com Python · Tema 17

Notebook de análise exploratória da economia brasileira a partir de uma base
simulada de indicadores trimestrais. Acompanha o dashboard interativo
(`app.py`) e a página de apresentação (`index.html`).

**Bibliotecas:** Pandas · NumPy · Matplotlib · Seaborn · SQLAlchemy (SQLite)


## 1. Introdução ao problema

Indicadores econômicos resumem o desempenho de um país e orientam decisões de
governos, empresas e investidores. Este trabalho investiga a evolução de
indicadores brasileiros entre **2015 e 2024** buscando responder:

- Como o **PIB** evoluiu ao longo do tempo?
- Existe relação entre **inflação e desemprego**?
- Quais períodos apresentaram **maior crescimento**?
- Há impacto da **taxa de juros sobre o consumo**?
- Quais indicadores foram **mais instáveis**?
- Como a **renda média** evoluiu e houve **períodos críticos**?


## 2. Contextualização econômica

O período 2015–2024 concentra eventos marcantes da economia brasileira: a
recessão de 2015–2016, a lenta recuperação seguinte, o choque da pandemia em
2020 e o posterior ciclo de alta de juros para conter a inflação. Os
indicadores aqui analisados — PIB, inflação, Selic, desemprego, câmbio, renda,
consumo, investimento e balança comercial — são justamente os que capturam
esses movimentos.

> **Observação importante:** a base utilizada é **simulada**. As conclusões
> servem para exercitar a metodologia de análise, não para descrever a
> realidade histórica com precisão.


## 3. Explicação da base de dados

Arquivo: `dados/simulacao_indicadores_economicos_brasil.csv` — 40 registros
(10 anos × 4 trimestres) e 14 colunas.

| Coluna | Descrição |
|---|---|
| `ano`, `trimestre`, `data` | Período de referência |
| `pib` | Produto Interno Bruto (R$ milhões) |
| `inflacao` | Índice de inflação (%) |
| `taxa_juros` | Taxa Selic (%) |
| `taxa_desemprego` | Desemprego (%) |
| `cambio_dolar` | Cotação do dólar (R$/US$) |
| `renda_media` | Renda média (R$) |
| `consumo_familias`, `investimento` | Índices de consumo e investimento |
| `exportacoes`, `importacoes` | Volumes de comércio exterior |
| `nivel_economico` | Crise, Estabilidade ou Crescimento |


## 4. Leitura dos dados

In [ ]:
import sys
sys.path.append("..")  # permite importar data_utils a partir de notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data_utils as du

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4.5)

bruto = du.carregar_dados(du.CSV_PADRAO)
print("Dimensões:", bruto.shape)
bruto.head()

## 5. Limpeza e preparação

Verificamos tipos, valores nulos e duplicatas. A função `carregar_dados`
já normaliza nomes de coluna (remoção de BOM/espaços), converte tipos,
remove duplicatas e ordena por data.


In [ ]:
# Diagnóstico de qualidade
du.diagnostico_qualidade(bruto)

In [ ]:
# Conferências rápidas
print("Duplicatas:", bruto.duplicated().sum())
print("Nulos totais:", int(bruto.isna().sum().sum()))
print("Período:", bruto["data"].min().date(), "->", bruto["data"].max().date())
bruto["nivel_economico"].value_counts()

## 6. Engenharia de atributos

Criamos atributos com significado econômico:

- **saldo_comercial** = exportações − importações;
- **indice_miseria** = inflação + desemprego (indicador clássico de bem-estar);
- **pib_var_tri** e **pib_var_anual** = variação do PIB trimestre a trimestre e ano a ano;
- **consumo_sobre_renda** = razão consumo/renda.


In [ ]:
df = du.preparar_dados(bruto)
df[["ano_tri", "pib", "pib_var_anual", "saldo_comercial",
    "indice_miseria", "nivel_economico"]].head(8)

## 7. KPIs — indicadores-chave

In [ ]:
kpis = du.calcular_kpis(df)
print(f"PIB médio............: R$ {kpis['pib_medio']/1e6:,.2f} milhões")
print(f"Inflação média.......: {kpis['inflacao_media']:.2f}%")
print(f"Desemprego médio.....: {kpis['desemprego_medio']:.2f}%")
print(f"Taxa de juros média..: {kpis['juros_medio']:.2f}%")
print(f"Dólar médio..........: R$ {kpis['dolar_medio']:.2f}")
print(f"Crescimento médio a/a: {kpis['crescimento_medio']:.2f}%")

## 8. Análise exploratória e visualizações

### 8.1 Evolução temporal do PIB


In [ ]:
cores = {"Crise": "#C65D4B", "Estabilidade": "#E8A33D", "Crescimento": "#3BA89A"}

fig, ax = plt.subplots()
ax.plot(df["data"], df["pib"], color="#0E1420", lw=2, zorder=2)
for nivel, cor in cores.items():
    s = df[df["nivel_economico"] == nivel]
    ax.scatter(s["data"], s["pib"], color=cor, s=55, zorder=3,
               edgecolor="white", label=nivel)
ax.set_title("Evolução do PIB trimestral (2015–2024)")
ax.set_ylabel("PIB (R$ milhões)")
ax.legend(title="Nível econômico", ncol=3)
plt.show()

### 8.2 Relação inflação × desemprego

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="inflacao", y="taxa_desemprego",
                hue="nivel_economico", palette=cores, s=80, ax=ax)
sns.regplot(data=df, x="inflacao", y="taxa_desemprego", scatter=False,
            color="#8A93A6", line_kws={"ls": "--"}, ax=ax)
r = df["inflacao"].corr(df["taxa_desemprego"])
ax.set_title(f"Inflação x Desemprego (r = {r:.2f})")
plt.show()

### 8.3 Matriz de correlação

In [ ]:
corr = du.matriz_correlacao(df)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            square=True, linewidths=.5, annot_kws={"size": 8}, ax=ax)
ax.set_title("Matriz de correlação dos indicadores")
plt.show()

### 8.4 Heatmap trimestral do PIB

In [ ]:
piv = df.pivot_table(index="ano", columns="trimestre", values="pib")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(piv/1e6, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=.5,
            cbar_kws={"label": "PIB (R$ milhões/1e6)"}, ax=ax)
ax.set_title("PIB por ano e trimestre")
plt.show()

### 8.5 Indicadores médios por nível econômico (barras comparativas)

In [ ]:
med = df.groupby("nivel_economico")[["inflacao", "taxa_desemprego",
      "taxa_juros"]].mean().reindex(["Crise", "Estabilidade", "Crescimento"])
med.plot(kind="bar", figsize=(9, 4.5),
         color=["#C65D4B", "#E8A33D", "#3BA89A"])
plt.title("Médias por nível econômico")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.legend(title="Indicador")
plt.show()

### 8.6 Persistência em banco (SQLAlchemy + SQLite) e consulta com JOIN

In [ ]:
caminho = du.criar_banco(df)
print("Banco criado em:", caminho)

sql = '''
    SELECT n.nivel_economico, COUNT(*) AS trimestres,
           ROUND(AVG(i.pib)/1e6, 2) AS pib_medio_milhoes
    FROM indicadores i
    JOIN niveis n ON i.id_nivel = n.id_nivel
    GROUP BY n.nivel_economico
    ORDER BY pib_medio_milhoes DESC
'''
du.consultar_banco(sql)

## 9. Interpretação dos resultados

- **Evolução do PIB:** por ser uma série simulada, o PIB oscila bastante entre
  trimestres, sem a tendência suave típica de dados reais. Ainda assim é
  possível identificar os trimestres de maior e menor atividade e associá-los
  aos rótulos de nível econômico.
- **Inflação × desemprego:** a correlação observada é **fraca**. Em dados
  reais, a teoria (curva de Phillips) sugere relação inversa de curto prazo;
  aqui o sinal é tênue por conta da natureza simulada da base.
- **Juros × consumo:** também sem relação linear forte neste recorte — em
  dados reais, esperaríamos que juros mais altos reduzissem o consumo.
- **Comparação por nível:** agrupar por `nivel_economico` e cruzar com as
  médias (via `GROUP BY`/`JOIN`) é a forma mais clara de contrastar os
  regimes de crise, estabilidade e crescimento.


## 10. Conclusão

O notebook percorreu o fluxo completo de um projeto de dados: leitura,
limpeza, engenharia de atributos, KPIs, visualizações, correlação e
persistência em banco relacional. Mais do que gerar gráficos, o objetivo foi
**estruturar uma aplicação analítica reprodutível**: a mesma lógica
(`data_utils.py`) alimenta este notebook e o dashboard Streamlit.

Como a base é simulada, as relações estatísticas aparecem fracas; a
metodologia, porém, é diretamente transferível para fontes oficiais como
**IBGE** (PIB, desemprego), **Banco Central** (Selic, câmbio) e **IPEA**.

**Próximos passos:** conectar a dados reais via API, adicionar previsão de
séries temporais e publicar o painel para acompanhamento contínuo.
